# 05 — Pooling vs summary

**Concept:** one document, embedded three ways. Title alone, title plus body as a
single string, and body split into chunks whose vectors are mean-pooled back into one.

## The prediction, written before running

> **Pooling helps.** The document body carries signal the title does not. A
> representation built from the body should beat one built from the title.
>
> **The adapter nulls.** Query and document pass through the same encoder, so they
> already live in one space. An adapter exists to fix cross-space misalignment, and
> here there is none to fix. That argument does not depend on the corpus, so it should
> transfer from the source system.

Both halves were committed to the repo's build plan before this notebook ran. The
adapter half replicates a null the source system had already measured; it is not an
independent bet.

## Mechanisms in play

"Pooling helps" is two comparisons with two different mechanisms.

1. `title-only` against the `dense` baseline. The baseline embeds `title + text`, so it
   reads the body already. This comparison tests the pre-registered mechanism head on.
2. `chunk-pool` against the `dense` baseline. Both read the body. What differs is
   chunking, mean-pooling, and the title weighting. The candidate mechanism is
   truncation escape. `all-MiniLM-L6-v2` accepts 256 tokens and NFCorpus abstracts run
   longer, so the one-string embed drops a tail that the chunks keep. A later cell
   measures how often that happens.

The source system saw a large lift from pooling: recall@100 went 0.37 to 0.55. Its firm
vector contained no portfolio text at all, so pooling handed it a new evidence source.
Our baseline already contains the body. Comparison 2 therefore starts from a much
smaller gap than the source result would suggest.

In [1]:
import numpy as np

from ragexp.adapter import apply_adapter, train_adapter
from ragexp.data import load_nfcorpus
from ragexp.embed import Embedder
from ragexp.metrics import paired_bootstrap
from ragexp.retrieve import _chunks, pool_doc_matrix
from ragexp.runs import evaluate_run, load_scores, metric_vector, save_scores, summarize

corpus = load_nfcorpus()
qids = [q.query_id for q in corpus.queries]
doc_ids = corpus.doc_ids
emb = Embedder()

dense_scores = load_scores("dense")  # the committed nb 02 baseline
print(corpus.summary())
print(f"docs with an empty title: {sum(1 for d in corpus.docs if not d.title.strip())}")

NFCorpus: 3633 docs, 323 queries, 12334 judgments (38.2/query), grade distribution {1: 11758, 2: 576}
docs with an empty title: 0


## Three representations

`pool_doc_matrix()` builds the `(3633, 384)` document matrix under each one.

- `title` embeds `d.title` on its own. This is the summary representation.
- `full` embeds `d.full`, which is `title + "\n\n" + text`. This is the nb 02 baseline.
- `pool` cuts `d.text` into chunks with `_chunks()` (400 characters, 80 overlap),
  embeds every chunk, takes the **mean**, then blends `0.25 * title + 0.75 *
  mean(chunks)` and re-normalises.

Mean, not max-sim. The port template `embed_pool_experiment.py` mean-pools a firm's
portfolio-company vectors and ranks against `0.25 * firm + 0.75 * pool`, the variant it
labels `blend_0.25`. `pool_doc_matrix(mode="pool")` is that arithmetic with chunks
standing in for portfolio companies. Holding the formula identical is what lets a lift
or a null here transfer as a claim about the method.

Max-sim is a different experiment. It keeps every chunk vector and scores a document by
its single best-matching chunk, which needs an index of one row per chunk rather than
one row per document. The template never used it, so neither do we.

In [2]:
title_matrix = pool_doc_matrix(corpus.docs, emb, mode="title")
full_matrix = pool_doc_matrix(corpus.docs, emb, mode="full")
pool_matrix = pool_doc_matrix(corpus.docs, emb, mode="pool")

n_chunks = [len(_chunks(d.text)) for d in corpus.docs]
print(f"title / full / pool matrices: {title_matrix.shape}, {full_matrix.shape}, {pool_matrix.shape}")
print(f"chunks: {sum(n_chunks)} total, median {int(np.median(n_chunks))} per doc, max {max(n_chunks)}")

title / full / pool matrices: (3633, 384), (3633, 384), (3633, 384)
chunks: 18835 total, median 5 per doc, max 32


## The one-string embed gets clipped

`d.full` enters the encoder as a single sequence. `all-MiniLM-L6-v2` sets
`max_seq_length = 256`, and every token past that is dropped before the model reads it.

Measure it rather than assume it.

In [3]:
tokenizer = emb._load().tokenizer
max_len = emb._load().max_seq_length
full_tokens = np.array([len(tokenizer.encode(d.full)) for d in corpus.docs])
chunk_tokens = np.array([len(tokenizer.encode(c)) for d in corpus.docs[:300] for c in _chunks(d.text)])

print(f"max_seq_length            : {max_len}")
print(f"tokens in d.full          : median {int(np.median(full_tokens))}, p90 {int(np.percentile(full_tokens, 90))}, max {full_tokens.max()}")
print(f"docs clipped by the encoder: {(full_tokens > max_len).mean():.1%}")
print(f"tokens per chunk (300 docs): median {int(np.median(chunk_tokens))}, max {chunk_tokens.max()}, clipped {(chunk_tokens > max_len).mean():.1%}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (386 > 256). Running this sequence through the model will result in indexing errors


max_seq_length            : 256
tokens in d.full          : median 357, p90 504, max 2377
docs clipped by the encoder: 78.8%
tokens per chunk (300 docs): median 86, max 200, clipped 0.0%


## Rank and score

One matmul per representation ranks all 3,633 documents for all 323 judged queries.
Scoring goes through `ragexp.runs`, so an undefined metric stays `None` and never
averages in as 0.0.

The `full` matrix is recomputed here as a pipeline check. Comparisons pair against the
committed `results/scores/dense.json`, not against this recomputation.

In [4]:
query_vecs = emb.encode([q.text for q in corpus.queries])  # unit vectors


def rank_all(doc_matrix, qv=None):
    # Rank every doc for every query: one (323, 3633) similarity matrix.
    sims = (query_vecs if qv is None else qv) @ doc_matrix.T
    return {qid: [doc_ids[j] for j in np.argsort(-sims[i])] for i, qid in enumerate(qids)}


rankings, run_scores = {}, {}
for name, matrix in [("title-only", title_matrix), ("full", full_matrix), ("chunk-pool", pool_matrix)]:
    rankings[name] = rank_all(matrix)
    run_scores[name] = evaluate_run(corpus, rankings[name], recall_ks=(10, 50), ndcg_ks=(10,))

save_scores("title_only", run_scores["title-only"])
save_scores("chunk_pool", run_scores["chunk-pool"])

header = f"{'system':<12}" + "".join(f"{m:>12}" for m in ("recall@10", "recall@50", "ndcg@10"))
print(header)
print("-" * len(header))
for name in ("title-only", "full", "chunk-pool"):
    s = summarize(run_scores[name])
    print(f"{name:<12}" + "".join(f"{s[m]:>12.4f}" for m in ("recall@10", "recall@50", "ndcg@10")))
committed = summarize(dense_scores)
print(f"{'dense (nb02)':<12}" + "".join(f"{committed[m]:>12.4f}" for m in ("recall@10", "recall@50", "ndcg@10")))

system         recall@10   recall@50     ndcg@10
------------------------------------------------
title-only        0.1355      0.2378      0.2885
full              0.1550      0.2508      0.3165
chunk-pool        0.1536      0.2455      0.3140
dense (nb02)      0.1550      0.2508      0.3167


### Pipeline check

`full` here and `dense` from nb 02 are the same representation reached by two code
paths. nb 02 embeds one query at a time through `retrieve.dense()`; this notebook
embeds all 323 in one batch. Padding differs between those two calls, so the
similarities differ in the last few decimal places and a handful of near-ties swap.
Anything larger than that is a bug.

In [5]:
for metric in ("recall@10", "recall@50", "ndcg@10"):
    gap = summarize(run_scores["full"])[metric] - committed[metric]
    print(f"{metric:>10}: recomputed - committed = {gap:+.5f}")

 recall@10: recomputed - committed = +0.00000
 recall@50: recomputed - committed = +0.00000
   ndcg@10: recomputed - committed = -0.00025


## One query the title cannot answer

The means hide the mechanism. The cell below finds a query where `title-only` drops a
relevant document out of the top 10 while `chunk-pool` keeps it there. Of all such
cases it takes the one with the most query terms present in the body and absent from
the title.

In [6]:
by_id = {d.doc_id: d for d in corpus.docs}


def body_only_terms(query, doc):
    # Query words (5+ chars) found in the body text and absent from the title.
    strip = ".,?:;()"
    q_words = {w.strip(strip).lower() for w in query.split() if len(w) > 4}
    title_words = {w.strip(strip).lower() for w in doc.title.split()}
    body_words = {w.strip(strip).lower() for w in doc.text.split()}
    return (q_words & body_words) - title_words


best = None
for q in corpus.queries:
    relevant = set(corpus.relevant_ids(q.query_id))
    gained = (set(rankings["chunk-pool"][q.query_id][:10]) & relevant) - set(rankings["title-only"][q.query_id][:10])
    for doc_id in sorted(gained):
        terms = body_only_terms(q.text, by_id[doc_id])
        if best is None or len(terms) > best[0]:
            best = (len(terms), q, by_id[doc_id])

_, q, doc = best
print(f"query          : {q.text}")
print(f"doc title      : {doc.title}")
print(f"grade          : {corpus.qrels[q.query_id][doc.doc_id]}")
print(f"body-only terms: {sorted(body_only_terms(q.text, doc))}")
print(f"rank under title-only: {rankings['title-only'][q.query_id].index(doc.doc_id) + 1}")
print(f"rank under chunk-pool: {rankings['chunk-pool'][q.query_id].index(doc.doc_id) + 1}")
print(f"\nbody, first 300 chars:\n{doc.text[:300]}")

query          : Stopping Heart Disease in Childhood
doc title      : Pathobiological determinants of atherosclerosis in youth risk scores are associated with early and advanced atherosclerosis.
grade          : 2
body-only terms: ['childhood', 'disease', 'heart']
rank under title-only: 29
rank under chunk-pool: 10

body, first 300 chars:
OBJECTIVES: Atherosclerosis begins in childhood and progresses during adolescence and young adulthood. The Pathobiological Determinants of Atherosclerosis in Youth Study previously reported risk scores to estimate the probability of advanced atherosclerotic lesions in young individuals aged 15 to 34


## The scoreboard

Each row is `paired_bootstrap` on per-query differences: 10,000 resamples over queries,
95% CI, two-sided p. A positive delta means the second system scored higher.

In [7]:
def compare(name_a, scores_a, name_b, scores_b, keys=None):
    keys = qids if keys is None else keys
    print(f"{name_b} vs {name_a}   (n={len(keys)} queries)")
    for metric in ("recall@10", "recall@50", "ndcg@10"):
        a = metric_vector(scores_a, metric, keys)
        b = metric_vector(scores_b, metric, keys)
        delta, (lo, hi), p = paired_bootstrap(a, b)
        sig = "  <-- significant" if (lo > 0 or hi < 0) else ""
        print(f"  {metric:>10}: delta {delta:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}{sig}")
    print()


compare("dense", dense_scores, "title-only", run_scores["title-only"])
compare("dense", dense_scores, "chunk-pool", run_scores["chunk-pool"])
compare("title-only", run_scores["title-only"], "chunk-pool", run_scores["chunk-pool"])

title-only vs dense   (n=323 queries)
   recall@10: delta -0.0195  CI [-0.0373, -0.0020]  p=0.0276  <-- significant
   recall@50: delta -0.0130  CI [-0.0279, +0.0022]  p=0.0954
     ndcg@10: delta -0.0282  CI [-0.0458, -0.0114]  p=0.0016  <-- significant

chunk-pool vs dense   (n=323 queries)
   recall@10: delta -0.0013  CI [-0.0116, +0.0103]  p=0.7632
   recall@50: delta -0.0053  CI [-0.0147, +0.0051]  p=0.2820
     ndcg@10: delta -0.0028  CI [-0.0145, +0.0093]  p=0.6312

chunk-pool vs title-only   (n=323 queries)
   recall@10: delta +0.0181  CI [+0.0037, +0.0341]  p=0.0138  <-- significant
   recall@50: delta +0.0077  CI [-0.0044, +0.0201]  p=0.2204
     ndcg@10: delta +0.0255  CI [+0.0116, +0.0399]  p=0.0002  <-- significant



### Where truncation should have paid

The `chunk-pool` vs `dense` null averages over queries that differ in how much text the
encoder threw away. Two effects can cancel inside that average.

- Truncation escape. `chunk-pool` reads the tail past token 256 and `dense` does not.
  This should favour `chunk-pool` when the relevant documents are long.
- Pooling dilution. A mean over five chunks averages the one chunk a query matches with
  four it does not. This should favour `dense` everywhere.

Split the queries by how often their own relevant documents get clipped, then pair
inside each tercile.

In [8]:
clipped = {d.doc_id: bool(full_tokens[i] > max_len) for i, d in enumerate(corpus.docs)}
clip_frac = {}
for q in corpus.queries:
    rel = [d for d in corpus.relevant_ids(q.query_id) if d in clipped]
    clip_frac[q.query_id] = float(np.mean([clipped[d] for d in rel])) if rel else None

defined = [q for q in qids if clip_frac[q] is not None]
lo_cut, hi_cut = np.percentile([clip_frac[q] for q in defined], [33.3, 66.7])
terciles = {
    "least clipped": [q for q in defined if clip_frac[q] <= lo_cut],
    "middle": [q for q in defined if lo_cut < clip_frac[q] <= hi_cut],
    "most clipped": [q for q in defined if clip_frac[q] > hi_cut],
}
print(f"tercile cuts on the clipped fraction: {lo_cut:.3f}, {hi_cut:.3f}\n")
for label, keys in terciles.items():
    mean_frac = np.mean([clip_frac[q] for q in keys])
    print(f"{label}: n={len(keys)}, mean fraction of relevant docs clipped = {mean_frac:.3f}")
    compare("dense", dense_scores, "chunk-pool", run_scores["chunk-pool"], keys=keys)

tercile cuts on the clipped fraction: 0.784, 0.874

least clipped: n=108, mean fraction of relevant docs clipped = 0.633
chunk-pool vs dense   (n=108 queries)
   recall@10: delta -0.0225  CI [-0.0401, -0.0074]  p=0.0010  <-- significant
   recall@50: delta -0.0199  CI [-0.0377, -0.0044]  p=0.0100  <-- significant
     ndcg@10: delta -0.0261  CI [-0.0477, -0.0059]  p=0.0100  <-- significant

middle: n=107, mean fraction of relevant docs clipped = 0.825
chunk-pool vs dense   (n=107 queries)
   recall@10: delta -0.0007  CI [-0.0106, +0.0098]  p=0.8722


   recall@50: delta -0.0022  CI [-0.0116, +0.0073]  p=0.6526


     ndcg@10: delta -0.0087  CI [-0.0256, +0.0079]  p=0.3080

most clipped: n=108, mean fraction of relevant docs clipped = 0.956
chunk-pool vs dense   (n=108 queries)
   recall@10: delta +0.0191  CI [-0.0017, +0.0490]  p=0.1076
   recall@50: delta +0.0063  CI [-0.0126, +0.0310]  p=0.6248
     ndcg@10: delta +0.0265  CI [+0.0067, +0.0520]  p=0.0058  <-- significant



## The adapter

The free representations are done. The trained one comes next.

`ragexp/adapter.py` ports `embed_adapter_train.py` from the source system. One
`384 x 384` matrix `W` sits on the query side, so a query scores a document as
`(W q) . d`. `W` starts at the identity, which makes `W = I` reproduce the `chunk-pool`
ranking exactly. Any gain has to be earned by training.

Two departures from the template, both stated so you can judge them.

- The template puts `W` on the document side and leaves the query untouched, so a new
  pitch needs no retraining. Ours sits on the query side, so a new document needs no
  re-encoding of the corpus. The scoring form is the same bilinear product either way.
- The template supervises with company-to-firm ownership, a signal independent of its
  eval labels. NFCorpus offers no equivalent free signal. Supervision here is the qrels
  themselves, which is what makes a query split mandatory.

**The split.** `np.random.default_rng(0).permutation` over the 323 query ids. The first
107 are held out, the remaining 216 train. Documents are shared across both sides,
queries are not. A training pair is one train query paired with one of its relevant
documents.

**Leakage guard.** The template drops training rows whose cosine to any eval item
exceeds 0.9. The same guard runs here over query vectors.

In [9]:
row_of_doc = {d: i for i, d in enumerate(doc_ids)}
perm = np.random.default_rng(0).permutation(len(qids))
n_eval = len(qids) // 3
eval_idx, train_idx = sorted(perm[:n_eval]), sorted(perm[n_eval:])
eval_qids = [qids[i] for i in eval_idx]

cross = query_vecs[train_idx] @ query_vecs[eval_idx].T
max_cos = cross.max(axis=1)
kept_idx = [i for i, m in zip(train_idx, max_cos) if m <= 0.9]
print(f"split          : {len(train_idx)} train / {len(eval_qids)} eval queries")
print(f"max train-eval query cosine: {max_cos.max():.4f}")
print(f"dropped for leakage        : {len(train_idx) - len(kept_idx)}  (train queries kept: {len(kept_idx)})")

pair_q, pair_target = [], []
for i in kept_idx:
    for d in corpus.relevant_ids(qids[i]):
        if d in row_of_doc:
            pair_q.append(query_vecs[i])
            pair_target.append(row_of_doc[d])
pair_q = np.vstack(pair_q).astype(np.float32)
pair_target = np.array(pair_target)
print(f"training pairs : {len(pair_target)}")

split          : 216 train / 107 eval queries
max train-eval query cosine: 0.9662
dropped for leakage        : 1  (train queries kept: 215)
training pairs : 8089


### Hyperparameters fixed from train loss only

The template fixes `lr`, `epochs` and weight decay a priori, then reports eval once. We
sweep `lr` over three values and read the train loss alone. The eval split is scored
after the choice is locked.

In [10]:
sweep = {}
for lr in (0.5, 5.0, 50.0):
    _, hist = train_adapter(pair_q, pool_matrix, pair_target, epochs=60, lr=lr, wd=3e-2)
    sweep[lr] = hist
    print(f"lr={lr:>5}: train loss {hist[0].loss:.4f} -> {hist[-1].loss:.4f}"
          f" | train acc@1 {hist[0].acc_at_1:.4f} -> {hist[-1].acc_at_1:.4f}"
          f" | ||W-I|| {hist[-1].drift:.3f}")

best_lr = min(sweep, key=lambda k: sweep[k][-1].loss)
print(f"\nchosen on train loss: lr={best_lr}")

lr=  0.5: train loss 8.1175 -> 8.1004 | train acc@1 0.0109 -> 0.0114 | ||W-I|| 0.689


lr=  5.0: train loss 8.1175 -> 8.0973 | train acc@1 0.0109 -> 0.0111 | ||W-I|| 1.124


lr= 50.0: train loss 8.1175 -> 8.1233 | train acc@1 0.0109 -> 0.0100 | ||W-I|| 1.760

chosen on train loss: lr=5.0


In [11]:
W, history = train_adapter(pair_q, pool_matrix, pair_target, epochs=60, lr=best_lr, wd=3e-2)
identity = np.eye(pool_matrix.shape[1], dtype=np.float32)
assert np.allclose(apply_adapter(identity, query_vecs), query_vecs, atol=1e-6), "W=I must be a no-op"
print("self-check: W=I leaves the query vectors untouched")

for h in history:
    if h.epoch in (1, 5, 10, 20, 40, 60):
        print(f"  ep{h.epoch:>3}  loss={h.loss:.4f}  train acc@1={h.acc_at_1:.4f}  ||W-I||={h.drift:.4f}")
print(f"\nrandom-guess acc@1 would be {1 / len(doc_ids):.5f}; the identity already scores {history[0].acc_at_1:.4f}")

_, long_run = train_adapter(pair_q, pool_matrix, pair_target, epochs=200, lr=best_lr, wd=3e-2)
print(f"convergence check, 200 epochs: loss {long_run[-1].loss:.4f} vs {history[-1].loss:.4f} at 60,"
      f" ||W-I|| {long_run[-1].drift:.4f} vs {history[-1].drift:.4f}")

self-check: W=I leaves the query vectors untouched
  ep  1  loss=8.1175  train acc@1=0.0109  ||W-I||=0.0000
  ep  5  loss=8.1023  train acc@1=0.0112  ||W-I||=0.5644
  ep 10  loss=8.0982  train acc@1=0.0114  ||W-I||=0.8885
  ep 20  loss=8.0974  train acc@1=0.0111  ||W-I||=1.0817
  ep 40  loss=8.0973  train acc@1=0.0111  ||W-I||=1.1226
  ep 60  loss=8.0973  train acc@1=0.0111  ||W-I||=1.1240

random-guess acc@1 would be 0.00028; the identity already scores 0.0109


convergence check, 200 epochs: loss 8.0973 vs 8.0973 at 60, ||W-I|| 1.1240 vs 1.1240


### Eval, scored once

`W = I` against trained `W`, same document matrix, same 107 held-out queries. The train
split is printed alongside to show where the parameters went.

In [12]:
base_scores = evaluate_run(corpus, rank_all(pool_matrix, apply_adapter(identity, query_vecs)),
                           recall_ks=(10, 50), ndcg_ks=(10,))
adapter_scores = evaluate_run(corpus, rank_all(pool_matrix, apply_adapter(W, query_vecs)),
                              recall_ks=(10, 50), ndcg_ks=(10,))
save_scores("adapter", {q: adapter_scores[q] for q in eval_qids})

base_means, pool_means = summarize(base_scores), summarize(run_scores["chunk-pool"])
assert all(np.isclose(base_means[m], pool_means[m], atol=1e-6) for m in base_means), "W=I drifted from chunk-pool"
print("self-check: the W=I run reproduces the chunk-pool means")

train_qids = [qids[i] for i in train_idx]
for split_name, keys in (("train", train_qids), ("eval", eval_qids)):
    b = summarize({q: base_scores[q] for q in keys})
    a = summarize({q: adapter_scores[q] for q in keys})
    print(f"[{split_name}] " + "  ".join(
        f"{m}: {b[m]:.4f} -> {a[m]:.4f} ({a[m] - b[m]:+.4f})" for m in ("recall@10", "recall@50", "ndcg@10")))
print()

compare("chunk-pool (W=I)", {q: base_scores[q] for q in eval_qids},
        "adapter (trained W)", {q: adapter_scores[q] for q in eval_qids}, keys=eval_qids)

self-check: the W=I run reproduces the chunk-pool means
[train] recall@10: 0.1590 -> 0.1580 (-0.0010)  recall@50: 0.2444 -> 0.2510 (+0.0066)  ndcg@10: 0.3191 -> 0.3263 (+0.0072)
[eval] recall@10: 0.1428 -> 0.1397 (-0.0031)  recall@50: 0.2478 -> 0.2233 (-0.0245)  ndcg@10: 0.3037 -> 0.2974 (-0.0063)

adapter (trained W) vs chunk-pool (W=I)   (n=107 queries)
   recall@10: delta -0.0031  CI [-0.0110, +0.0027]  p=0.3864
   recall@50: delta -0.0245  CI [-0.0510, -0.0038]  p=0.0112  <-- significant
     ndcg@10: delta -0.0063  CI [-0.0184, +0.0050]  p=0.2804



## Resolving the prediction

### Half one, pooling helps: partly

Against the summary representation it holds. `chunk-pool` beats `title-only` on nDCG@10
by +0.0255 (p = 0.0002) and on recall@10 by +0.0181 (p = 0.0138). recall@50 comes back
a null at +0.0077, CI [-0.0044, +0.0201]. `title-only` also loses to the `dense`
baseline on nDCG@10 (-0.0282, p = 0.0016) and recall@10 (-0.0195, p = 0.0276). The
pre-registered mechanism survives: the body carries ranking signal the title lacks, and
it shows up first where ordering matters.

Against the `dense` baseline it fails. `chunk-pool` vs `dense` is a null on all three
metrics over the full 323 queries, every CI straddling zero, the largest delta being
-0.0053 on recall@50.

That corpus-level null is two effects cancelling. Split the queries by how often their
relevant documents get clipped and the sign flips.

- Least-clipped tercile: `chunk-pool` loses on all three metrics. nDCG@10 -0.0261
  (p = 0.0100), recall@10 -0.0225 (p = 0.0010), recall@50 -0.0199 (p = 0.0100).
- Most-clipped tercile: `chunk-pool` gains nDCG@10 by 0.0265 (p = 0.0058), with
  recall@10 at +0.0191 (p = 0.1076) and recall@50 a null.

Truncation escape is real, and so is the cost of averaging five chunks when a query
matches one of them. The first pays only where text was actually dropped. The second
charges everywhere. Over 323 queries they roughly cancel, which is why the headline row
reads as a null.

The source system's 0.37 to 0.55 lift stayed home. Its premise was that the firm vector
held zero portfolio text, so pooling introduced evidence the ranker had never seen.
Here `title + text` already held the body. Pooling only re-weighted evidence the
baseline already had.

### Half two, the adapter nulls: held

No metric moves in the adapter's favour on the held-out queries. recall@10 -0.0031
(p = 0.39), nDCG@10 -0.0063 (p = 0.28), recall@50 -0.0245 with CI [-0.0510, -0.0038]
and p = 0.0112, which is a significant loss.

Three pieces of evidence say this is convergence and not a training failure:

- Train loss moves 8.1175 to 8.0973 over 60 epochs. The starting point is `log(3633)`
  = 8.198, so the identity was already almost all of the achievable loss.
- Train acc@1 sits at 0.011 throughout, against 0.00028 for random guessing. The
  identity map already ranks a query's own relevant document first about 1% of the
  time, and training does not raise that.
- Running 200 epochs lands on the identical loss and the identical `||W-I||`. The
  optimiser reached a fixed point well before epoch 60.

Where the parameters did go is visible in the split-by-split print. nDCG@10 rises on
the train queries (0.3191 to 0.3263) and falls on the held-out ones (0.3037 to 0.2974).
147,456 parameters fitted to 215 queries memorised those queries. The mechanism claim
stands as written: same encoder, one space, nothing for a linear map to align.

One honest bound. A query-side `W` could in principle learn something that is not
alignment, such as dimensions that mark generally popular positives, and that would
transfer. It did not learn that here. A second bound: the eval split is 107 queries, so
the CIs are roughly 1.7x wider than the 323-query rows above.

## What goes in LESSONS.md

| what | number | verdict | lesson |
|---|---|---|---|
| `title-only` vs `dense` (title+text) | nDCG@10 -0.0282 (p=0.0016), recall@10 -0.0195 (p=0.0276), recall@50 -0.0130 (p=0.0954) | prediction held on nDCG@10 and recall@10, recall@50 null | embedding a summary in place of the document costs ranking quality first and recall later |
| `chunk-pool` vs `title-only` | nDCG@10 +0.0255 (p=0.0002), recall@10 +0.0181 (p=0.0138), recall@50 +0.0077 (null) | prediction held | mean-pooled body chunks recover most of what the title drops |
| `chunk-pool` vs `dense` (title+text) | recall@10 -0.0013, recall@50 -0.0053, nDCG@10 -0.0028, all three null | prediction failed across the full 323 queries | pooling buys nothing on average when the baseline string already contains the body: the source system's lift came from its firm vector holding no portfolio text |
| same comparison, queries split by encoder truncation | nDCG@10 -0.0261 (p=0.0100) in the least-clipped tercile, +0.0265 (p=0.0058) in the most-clipped | two effects cancelling | a null over a heterogeneous query set is the sum of its subgroups, so split before concluding a mechanism is absent |
| encoder truncation | 78.8% of docs exceed `max_seq_length=256` in `d.full` | measured, and it matters unevenly | four documents in five lose their tail before the model reads them, which changes the ranking only for queries whose relevant documents are the long ones |
| InfoNCE adapter vs `W=I`, 107 held-out queries | recall@10 -0.0031 (p=0.39), nDCG@10 -0.0063 (p=0.28), recall@50 -0.0245 (p=0.0112) | prediction held | a linear adapter fixes cross-space misalignment, and one encoder for both sides leaves none to fix |
| adapter train vs eval | nDCG@10 +0.0072 on 215 train queries, -0.0063 on 107 held-out | overfit, as expected | 147,456 parameters over 215 queries memorise the split; only the held-out number is evidence |

Next: notebook 06 takes learning to rank to MSLR-WEB10K, where the features are
pre-computed and Fold 1 trains on 6,000 queries rather than 215 (10,000 is the dataset total).